<h1 style="text-align:center;">Рекомендательная система для музыкального сервиса</h1>

# Overview

Постройте рекомендательную систему для музыкального сервиса: по истории событий пользователя предскажите, какие треки он будет слушать дальше. Для каждого пользователя нужно выдать до 10 треков, от лучшего к худшему.

# Description

В задании используются реальные логи музыкального сервиса: прослушивания, лайки и дизлайки. Каталог большой, больше 880 тысяч треков, поэтому оценить каждый трек для каждого пользователя дорого.

Рекомендовать уже прослушанные треки можно: повторные прослушивания считаются попаданием, как и в настоящем музыкальном сервисе.

Оценивание. Разбаловка будет опубликована отдельно.



# Evaluation

Метрика — NDCG@10, усреднённая по пользователям. Каждое попадание в топ-10 приносит тем больше, чем выше стоит трек, а результат нормируется на лучший возможный для этого пользователя.

Тестовые пользователи разделены на Public и Private примерно поровну. Во время конкурса виден скор Public, итоговое место определяется по Private.

# Dataset Description

## Данные

Основа: Yambda-50M от Яндекса, лицензия Apache-2.0.

Идентификаторы пользователей и треков перекодированы, время сдвинуто на постоянную величину. Использовать исходный датасет Yambda и сопоставлять с ним данные конкурса нельзя.

## Файлы

- train.parquet — события до момента разделения: 43.0 млн строк, 9844 пользователя.
- test_users.csv — 9213 пользователей, для которых нужны рекомендации.
- sample_submission.csv — пример сабмита со случайными треками.

## Колонки train.parquet

- `uid` — пользователь. Это тот же идентификатор, что user_id в CSV-файлах.
- `timestamp` — время события в секундах, со сдвигом. Настоящую дату по нему не определить, но порядок событий и интервалы сохранены.
- `item_id` — трек.
- `is_organic` — 1, если пользователь сам нашёл трек; 0, если трек пришёл из рекомендаций.
- `played_ratio_pct` — доля прослушанного трека в процентах. Может быть больше 100. Заполнена только для listen.
- `track_length_seconds` — длина трека в секундах. Заполнена только для listen.
-`event_type` — тип события: listen, like, unlike, dislike, undislike.

## Примечания

- Сплит по времени: граница проходит по 90% квантилю времени прослушиваний. Train содержит все типы событий до неё.
- Правильный ответ — треки, которые пользователь слушал после границы. Считается любое прослушивание, даже неполное.
- 151 тестового пользователя нет в train, для них нужен отдельный подход.

# 0. Импорты

In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [6]:
DATA_DIR = Path("data/raw")
TRAIN_PATH = DATA_DIR / "train.parquet"
TEST_PATH = DATA_DIR / "test_users.csv"

# 1. Подготовка данных к работе

## Загрузка данных

In [21]:
train = pd.read_parquet(TRAIN_PATH)

In [139]:
test_users = pd.read_csv(TEST_PATH)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


## Анализ данных

In [22]:
train.shape

(43000347, 7)

In [24]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 43000347 entries, 0 to 43000346
Data columns (total 7 columns):
 #   Column                Dtype   
---  ------                -----   
 0   uid                   uint32  
 1   timestamp             int64   
 2   item_id               uint32  
 3   is_organic            uint8   
 4   played_ratio_pct      float64 
 5   track_length_seconds  float64 
 6   event_type            category
dtypes: category(1), float64(2), int64(1), uint32(2), uint8(1)
memory usage: 1.4 GB


In [23]:
train.head()

,uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds,event_type
0,1,23850979,358453,0,96.0,145.0,listen
1,1,23851159,688547,0,100.0,180.0,listen
2,1,23851564,19131,0,100.0,245.0,listen
3,1,23851719,798591,0,100.0,160.0,listen
4,1,23851889,557996,0,100.0,170.0,listen


In [25]:
train.describe()

,uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds
count,4.300035e+07,4.300035e+07,4.300035e+07,4.300035e+07,4.182048e+07,4.182048e+07
mean,4.991880e+03,3.708203e+07,4.710566e+05,5.206316e-01,6.462953e+01,2.073706e+02
std,2.887943e+03,6.929920e+06,2.686429e+05,4.995742e-01,4.413305e+01,8.415808e+01
min,1.000000e+00,2.381809e+07,1.000000e+00,0.000000e+00,0.000000e+00,5.000000e+00
25%,2.479000e+03,3.128874e+07,2.389250e+05,0.000000e+00,7.000000e+00,1.650000e+02
50%,5.026000e+03,3.758673e+07,4.700480e+05,1.000000e+00,1.000000e+02,2.000000e+02
75%,7.451000e+03,4.322883e+07,7.051110e+05,1.000000e+00,1.000000e+02,2.350000e+02
max,1.000000e+04,4.794476e+07,9.340570e+05,1.000000e+00,1.590000e+02,2.495000e+03


In [26]:
train.isna().sum()

uid                           0
timestamp                     0
item_id                       0
is_organic                    0
played_ratio_pct        1179866
track_length_seconds    1179866
event_type                    0
dtype: int64

In [28]:
train["event_type"].value_counts()

event_type
listen       41820481
like           781435
unlike         281924
dislike         97411
undislike       19096
Name: count, dtype: int64

In [30]:
print("Уникальных пользователей:", train["uid"].nunique())
print("Уникальных треков:", train["item_id"].nunique())

Уникальных пользователей: 9844
Уникальных треков: 881514


Пользователей сильно меньше, чем треков -> матрица взаимодействий ожидаемо сильно разреженная

In [31]:
train_users = set(train["uid"].unique())
test_user_ids = set(test_users["user_id"])

known_users = train_users & test_user_ids
new_users = test_user_ids - train_users

print("Всего тестовых пользователей:", len(test_user_ids))
print("Пользователей с историей:", len(known_users))
print("Пользователей без истории:", len(new_users))

Всего тестовых пользователей: 9213
Пользователей с историей: 9062
Пользователей без истории: 151


151 тестовых пользователей нет в train - согласуется с тз

# 2. Разбиение данных на train/val

## Граница

По тз: 
"Сплит по времени: граница проходит по 90% квантилю времени прослушиваний. Train содержит все типы событий до неё."

Найдем этот 90% квантилю времени прослушиваний:

In [32]:
listen_mask = train["event_type"].eq("listen")
time_cutoff = train.loc[listen_mask, "timestamp"].quantile(0.9)

In [33]:
print("Временная граница:", time_cutoff)

Временная граница: 46146954.0


## Делим

In [34]:
train_split = train[train["timestamp"] <= time_cutoff].copy()
val_split = train.loc[
    (train["timestamp"] > time_cutoff) &
    (train["event_type"] == "listen"),
    ["uid", "item_id"]
].copy()

train_split содержит все типы событий до временной границы

val_split содержит только целевые прослушивания после границы

# 3. Popularity Baseline

В качестве бейзлайна возьмем рекомендацию всем пользователям самых поппулярных треков

Количество прослушиваний треков:

In [35]:
track_popularity = (
    train_split[train_split["event_type"] == "listen"]
    .groupby("item_id")
    .size()
    .sort_values(ascending=False)
)

Топ 10 треков:

In [36]:
top10_tracks = track_popularity.head(10).index.tolist()

In [37]:
top10_tracks

[894940, 337739, 433438, 776946, 543001, 642739, 535546, 221507, 58794, 472495]

Целевые прослушивания:

In [40]:
valid_relevant = (
    val_split.groupby("uid")["item_id"]
    .apply(set)
    .to_dict()
)

In [41]:
def ndcg_at_k(recommendations, relevant_items, k=10):
    relevant_items = set(relevant_items)

    if not relevant_items:
        return 0.0

    dcg = sum(
        1 / np.log2(rank + 1)
        for rank, item in enumerate(recommendations[:k], start=1)
        if item in relevant_items
    )

    ideal_hits = min(len(relevant_items), k)
    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )

    return dcg / idcg

In [42]:
baseline_scores = [
    ndcg_at_k(top10_tracks, relevant_items, k=10)
    for relevant_items in valid_relevant.values()
]

In [43]:
baseline_ndcg = np.mean(baseline_scores)
print(f"Popularity Baseline NDCG@10: {baseline_ndcg:.6f}")

Popularity Baseline NDCG@10: 0.150227


Отправная точка - NDCG@10 = 0.150227

# 4. Кандидатогенерация

Поскольку датасет не содержит описаний пользователей и музыкальных композиций, контентные методы в данной задаче применить затруднительно, поэтому будем рассматривать коллаборативную фильтрацию, использующая историю взаимодействий пользователей с треками

## Метрики: Recall@K, HitRate@K  

In [47]:
def recall_at_k(recommendations, relevant_items, k=100):
    relevant_items = set(relevant_items)

    if not relevant_items:
        return 0.0

    recommended = set(recommendations[:k])
    return len(recommended & relevant_items) / len(relevant_items)

In [49]:
def hit_rate_at_k(recommendations, relevant_items, k=100):
    relevant_items = set(relevant_items)

    if not relevant_items:
        return 0.0

    return int(bool(set(recommendations[:k]) & relevant_items))

## Матрица взаимодействий

In [55]:
from scipy.sparse import csr_matrix

In [56]:
def build_interaction_matrix(
    df,
    user_col="uid",
    item_col="item_id",
    event_col="event_type"
):
    # количество прослушиваний каждой пары пользователь–трек
    interactions = (
        df[df[event_col] == "listen"]
        .groupby([user_col, item_col])
        .size()
        .reset_index(name="count")
    )

    user_idx, user_ids = pd.factorize(interactions[user_col])
    item_idx, item_ids = pd.factorize(interactions[item_col])

    R = csr_matrix(
        (
            interactions["count"].to_numpy(dtype=np.float32),
            (user_idx, item_idx)
        ),
        shape=(len(user_ids), len(item_ids))
    )

    return R, user_ids, item_ids

In [57]:
R, user_ids, item_ids = build_interaction_matrix(train_split)

print("Размер матрицы:", R.shape)
print("Количество ненулевых элементов:", R.nnz)
print("Плотность:", R.nnz / (R.shape[0] * R.shape[1]))

Размер матрицы: (8711, 781188)
Количество ненулевых элементов: 10877202
Плотность: 0.001598429980779925


## iALS

Поскольку в датасете преобладают прослушивания, а не явные оценки треков, попробуем iALS, предназначенный для работы с неявной обратной связью

In [59]:
from implicit.als import AlternatingLeastSquares

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


В качестве начальной конфигурации обучим iALS с параметрами factors=64, alpha=20 и оценим качество полученных рекомендаций по метрикам Recall@K и HitRate@K

In [62]:
def fit_ials(
    R,
    factors=64,
    regularization=0.01,
    iterations=20,
    alpha=20,
    num_threads=4
):
    model = AlternatingLeastSquares(
        factors=factors,
        regularization=regularization,
        iterations=iterations,
        alpha=alpha,
        num_threads=num_threads,
        random_state=2501
    )

    model.fit(R)

    return model

In [63]:
ials_model = fit_ials(
    R,
    factors=64,
    regularization=0.01,
    iterations=20,
    alpha=20
)

100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [01:21<00:00,  4.09s/it]


Генкрация кандидатов для юзера:

In [64]:
def generate_ials_candidates(
    model,
    R,
    user_idx,
    n_candidates=300
):
    item_indices, scores = model.recommend(
        userid=user_idx,
        user_items=R[user_idx],
        N=n_candidates,
        filter_already_liked_items=False
    )

    return item_indices, scores

Генерация реком-ий для всех юзеров:

In [65]:
def generate_ials_recommendations(
    model,
    R,
    user_ids,
    user_to_idx,
    n_candidates=300
):
    recommendations = {}

    for uid in user_ids:
        if uid not in user_to_idx:
            recommendations[uid] = []
            continue

        user_idx = user_to_idx[uid]
        item_indices, scores = generate_ials_candidates(
            model,
            R,
            user_idx,
            n_candidates=n_candidates
        )
        recommendations[uid] = [
            (item_ids[item_idx], float(score))
            for item_idx, score in zip(item_indices, scores)
        ]

    return recommendations

Получим по 300 кандидатов для каждого пользователя с помощью обученной модели iALS:

In [66]:
ials_recommendations = generate_ials_recommendations(
    model=ials_model,
    R=R,
    user_ids=list(valid_relevant.keys()),
    user_to_idx={uid: idx for idx, uid in enumerate(user_ids)},
    n_candidates=300
)

Оценка качества кандген:

In [146]:
def evaluate_recommendations(
    recommendations,
    valid_relevant,
    candidate_sizes=(100, 300),
    method="iALS"
):
    results = []

    for k in candidate_sizes:
        recalls = []
        hit_rates = []

        for uid, relevant_items in valid_relevant.items():
            candidates = [
                item for item, score in recommendations.get(uid, [])
            ]

            recalls.append(
                recall_at_k(candidates, relevant_items, k)
            )

            hit_rates.append(
                hit_rate_at_k(candidates, relevant_items, k)
            )

        results.append({
            "method": method,
            "K": k,
            "Recall@K": np.mean(recalls),
            "HitRate@K": np.mean(hit_rates)
        })

    return pd.DataFrame(results)

In [68]:
ials_results = evaluate_recommendations(
    ials_recommendations,
    valid_relevant
)

ials_results

,method,K,Recall@K,HitRate@K
0,iALS,100,0.069474,0.861081
1,iALS,300,0.163635,0.907672


Сравним качество при разных альфа

In [69]:
alpha_values = [10, 20, 40]

ials_tuning_results = []

for alpha in alpha_values:
    print(f"Training iALS with alpha={alpha}")

    model = fit_ials(
        R,
        factors=64,
        regularization=0.01,
        iterations=20,
        alpha=alpha
    )

    recommendations = generate_ials_recommendations(
        model=model,
        R=R,
        user_ids=list(valid_relevant.keys()),
        user_to_idx={uid: idx for idx, uid in enumerate(user_ids)},
        n_candidates=300
    )

    results = evaluate_recommendations(
        recommendations,
        valid_relevant
    )

    results["alpha"] = alpha
    ials_tuning_results.append(results)

ials_tuning_results = pd.concat(
    ials_tuning_results,
    ignore_index=True
)

Training iALS with alpha=10


100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [01:21<00:00,  4.09s/it]


Training iALS with alpha=20


100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [01:22<00:00,  4.10s/it]


Training iALS with alpha=40


100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [01:22<00:00,  4.11s/it]


In [70]:
ials_tuning_results

,method,K,Recall@K,HitRate@K,alpha
0,iALS,100,0.077239,0.874131,10
1,iALS,300,0.176157,0.909013,10
2,iALS,100,0.069474,0.861081,20
3,iALS,300,0.163635,0.907672,20
4,iALS,100,0.059846,0.833028,40
5,iALS,300,0.145695,0.901573,40


=> лучший результат получен при alpha=10

Зафиксируем это значение для дальнейших экспериментов

In [71]:
best_alpha = 10

Теперь проверим влияние числа факторов на качество рекомендаций

In [72]:
factors_values = [32, 64, 128]

ials_factors_results = []

for factors in factors_values:
    print(f"Training iALS with factors={factors}")

    model = fit_ials(
        R,
        factors=factors,
        regularization=0.01,
        iterations=20,
        alpha=best_alpha
    )
    recommendations = generate_ials_recommendations(
        model=model,
        R=R,
        user_ids=list(valid_relevant.keys()),
        user_to_idx={uid: idx for idx, uid in enumerate(user_ids)},
        n_candidates=300
    )
    results = evaluate_recommendations(
        recommendations,
        valid_relevant
    )

    results["factors"] = factors
    ials_factors_results.append(results)

Training iALS with factors=32


100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [00:23<00:00,  1.20s/it]


Training iALS with factors=64


100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [01:21<00:00,  4.10s/it]


Training iALS with factors=128


100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [02:54<00:00,  8.72s/it]


In [73]:
ials_factors_results = pd.concat(
    ials_factors_results,
    ignore_index=True
)
ials_factors_results

,method,K,Recall@K,HitRate@K,factors
0,iALS,100,0.073569,0.875473,32
1,iALS,300,0.161587,0.906574,32
2,iALS,100,0.077239,0.874131,64
3,iALS,300,0.176157,0.909013,64
4,iALS,100,0.082526,0.878766,128
5,iALS,300,0.194702,0.912794,128


=> лучший результат получен при alpha=10, фиксируем

In [74]:
best_factor = 128

Зафиксируем выбранные гиперпараметры iALS

In [79]:
ials_best_config = {
    "factors": 128,
    "alpha": 10,
    "regularization": 0.01,
    "iterations": 20
}

Обучим итоговую модель с выбранными гиперпараметрами:

In [142]:
best_ials_model = fit_ials(
    R,
    **ials_best_config
)

100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [05:06<00:00, 15.34s/it]


Сформируем рекомендации для пользователей валидационной выборки:

In [143]:
ials_recommendations_best = generate_ials_recommendations(
    model=best_ials_model,
    R=R,
    user_ids=list(valid_relevant.keys()),
    user_to_idx={uid: idx for idx, uid in enumerate(user_ids)},
    n_candidates=300
)

Оценим итоговые рекомендации по NDCG@10

In [144]:
ials_ndcg_scores = []

for uid, relevant_items in valid_relevant.items():
    recommendations = [
        item for item, score in ials_recommendations_best.get(uid, [])
    ]

    ials_ndcg_scores.append(
        ndcg_at_k(recommendations, relevant_items, k=10)
    )

ials_ndcg = np.mean(ials_ndcg_scores)

print(f"iALS NDCG@10: {ials_ndcg:.6f}")

iALS NDCG@10: 0.158706


## BPR

iALS восстанавливает матрицу взаимодействий пользователей с треками на основе истории прослушиваний

BPR не восстанавливает матрицу взаимодействий. Он обучается на парах треков: для каждого пользователя прослушанный трек должен получать более высокий рейтинг, чем непрослушанный, то есть модель учится упорядочивать треки по предпочтениям пользователей

Тк в соревновании используется NDCG@10, которая учитывает позиции релевантных треков, проверим, даст ли такой подход прирост

In [83]:
from implicit.bpr import BayesianPersonalizedRanking

Для обучения BPR нужна бинарная матрица взаимодействий, заменим все все ненулевые значения в R на 1:

In [84]:
R_binary = R.copy()
R_binary.data[:] = 1

In [85]:
bpr_model = BayesianPersonalizedRanking(
    factors=128,
    learning_rate=0.05,
    regularization=0.01,
    iterations=30,
    num_threads=4,
    random_state=42
)

bpr_model.fit(R_binary)

100%|█████████████████████████████████████████████████| 30/30 [06:07<00:00, 12.24s/it, train_auc=95.16%, skipped=7.64%]


In [86]:
def generate_bpr_recommendations(
    model,
    R,
    user_ids,
    item_ids,
    n_candidates=300
):
    user_to_idx = {
        uid: idx for idx, uid in enumerate(user_ids)
    }

    known_users = [
        uid for uid in valid_relevant
        if uid in user_to_idx
    ]

    user_indices = np.array(
        [user_to_idx[uid] for uid in known_users]
    )

    user_items = R[user_indices]

    item_indices, scores = model.recommend(
        userid=user_indices,
        user_items=user_items,
        N=n_candidates,
        filter_already_liked_items=False
    )

    recommendations = {}

    for row, uid in enumerate(known_users):
        recommendations[uid] = [
            (item_ids[item_idx], float(score))
            for item_idx, score in zip(
                item_indices[row],
                scores[row]
            )
        ]

    # Пользователи без истории в train
    for uid in valid_relevant:
        if uid not in recommendations:
            recommendations[uid] = []

    return recommendations

In [87]:
bpr_recommendations = generate_bpr_recommendations(
    model=bpr_model,
    R=R_binary,
    user_ids=user_ids,
    item_ids=item_ids,
    n_candidates=300
)

Оценим кандген:

In [148]:
bpr_results = evaluate_recommendations(
    bpr_recommendations,
    valid_relevant,
    method="BPR"
)

bpr_results

,method,K,Recall@K,HitRate@K
0,BPR,100,0.065344,0.859373
1,BPR,300,0.139011,0.896817


Модель BPR возвращает кандидатов, упорядоченных по оценкам релевантности. Рассчитаем NDCG@10 для первых десяти треков в этом списке, чтобы оценить, насколько высоко модель ставит релевантные объекты:

In [89]:
bpr_ndcg_scores = []

for uid, relevant_items in valid_relevant.items():
    recommendations = [
        item for item, score
        in bpr_recommendations.get(uid, [])
    ]

    bpr_ndcg_scores.append(
        ndcg_at_k(recommendations, relevant_items, k=10)
    )

bpr_ndcg = np.mean(bpr_ndcg_scores)

print(f"BPR NDCG@10: {bpr_ndcg:.6f}")

BPR NDCG@10: 0.206755


## iALS + BPR

 iALS и BPR могут находить разные треки, объединим их

Объединим списки кандидатов, полученные с помощью iALS и BPR. Для каждого пользователя соберём уникальные треки и сохраним оценки обеих моделей. Если трек найден только одной моделью, для второй модели его оценка будет равна нулю. Из общего списка оставим до 600 кандидатов

In [90]:
def merge_recommendations(
    rec1,
    rec2,
    n_candidates=1000
):
    merged = {}

    all_users = set(rec1) | set(rec2)

    for uid in all_users:
        scores = {}

        for item, score in rec1.get(uid, []):
            scores.setdefault(item, {})["ials"] = score

        for item, score in rec2.get(uid, []):
            scores.setdefault(item, {})["bpr"] = score

        candidates = []

        for item, model_scores in scores.items():
            candidates.append({
                "item_id": item,
                "ials_score": model_scores.get("ials", 0.0),
                "bpr_score": model_scores.get("bpr", 0.0)
            })

        candidates.sort(
            key=lambda x: (
                x["ials_score"] > 0,
                x["bpr_score"] > 0
            ),
            reverse=True
        )

        merged[uid] = candidates[:n_candidates]

    return merged

Получим объединённые списки кандидатов для всех пользователей и для каждого трека суммарную оценку двух моделей:

In [91]:
merged_candidates = merge_recommendations(
    ials_recommendations_best,
    bpr_recommendations,
    n_candidates=600
)

merged_recommendations = {
    uid: [
        (x["item_id"], x["ials_score"] + x["bpr_score"])
        for x in candidates
    ]
    for uid, candidates in merged_candidates.items()
}

Проверим, улучшает ли объединение списков iALS и BPR полноту генерации кандидатов. Для этого объединим списки без повторений и рассчитаем метрики:

In [92]:
def evaluate_union_recall(
    rec1,
    rec2,
    valid_relevant
):
    recalls = []
    hit_rates = []

    for uid, relevant_items in valid_relevant.items():
        items1 = {
            item for item, score in rec1.get(uid, [])
        }
        items2 = {
            item for item, score in rec2.get(uid, [])
        }

        # Объединение кандидатов без дубликатов
        candidates = items1 | items2

        relevant_items = set(relevant_items)

        recalls.append(
            len(candidates & relevant_items) / len(relevant_items)
            if relevant_items else 0.0
        )

        hit_rates.append(
            int(bool(candidates & relevant_items))
        )

    return {
        "method": "iALS + BPR union",
        "Recall": np.mean(recalls),
        "HitRate": np.mean(hit_rates)
    }

Метрики для объединённого списка кандидатов:

In [93]:
union_results = pd.DataFrame([
    evaluate_union_recall(
        ials_recommendations_best,
        bpr_recommendations,
        valid_relevant
    )
])

union_results

,method,Recall,HitRate
0,iALS + BPR union,0.242965,0.920722


Объединение дало прирост, фиксируем

## История прослушиваний

В качестве третьего источника кандидатов используем историю прослушиваний пользователей. Для каждого пользователя отберём наиболее часто прослушиваемые треки за последние 7 дней, 30 дней и за всё время. Это позволит дополнить рекомендации iALS и BPR треками из собственной истории пользователя.

In [103]:
def generate_history_candidates(
    train_split,
    user_ids,
    cutoff,
    n_7d=300,
    n_30d=150,
    n_all=100
):
    listens = train_split.loc[
        train_split["event_type"].eq("listen"),
        ["uid", "item_id", "timestamp"]
    ]

    # Количество прослушиваний за всё время
    all_counts = (
        listens.groupby(["uid", "item_id"])
        .size()
        .rename("cnt_all")
        .reset_index()
    )

    # Прослушивания за последние 7 дней
    recent_7d = listens.loc[
        listens["timestamp"] >= cutoff - 7 * 86400
    ]

    counts_7d = (
        recent_7d.groupby(["uid", "item_id"])
        .size()
        .rename("cnt_7d")
        .reset_index()
    )

    # Прослушивания за последние 30 дней
    recent_30d = listens.loc[
        listens["timestamp"] >= cutoff - 30 * 86400
    ]

    counts_30d = (
        recent_30d.groupby(["uid", "item_id"])
        .size()
        .rename("cnt_30d")
        .reset_index()
    )

    history = (
        all_counts
        .merge(counts_7d, on=["uid", "item_id"], how="left")
        .merge(counts_30d, on=["uid", "item_id"], how="left")
        .fillna({"cnt_7d": 0, "cnt_30d": 0})
    )

    # Ранги внутри каждого пользователя
    history["rank_7d"] = (
        history.groupby("uid")["cnt_7d"]
        .rank(method="first", ascending=False)
    )

    history["rank_30d"] = (
        history.groupby("uid")["cnt_30d"]
        .rank(method="first", ascending=False)
    )

    history["rank_all"] = (
        history.groupby("uid")["cnt_all"]
        .rank(method="first", ascending=False)
    )

    # Объединяем три источника исторических кандидатов
    history = history[
        (history["rank_7d"] <= n_7d)
        | (history["rank_30d"] <= n_30d)
        | (history["rank_all"] <= n_all)
    ].copy()

    # Сохраняем score и ранг для дальнейшего ранжирования
    history["hist_score"] = (
        np.log1p(history["cnt_7d"])
        + np.log1p(history["cnt_30d"])
        + np.log1p(history["cnt_all"])
    )

    history = history.sort_values(
        ["uid", "hist_score"],
        ascending=[True, False]
    )

    result = {}

    for uid, group in history.groupby("uid"):
        result[uid] = list(zip(
            group["item_id"].astype(int),
            group["hist_score"].astype(float)
        ))

    return result


hist_recommendations = generate_history_candidates(
    train_split=train_split,
    user_ids=user_ids,
    cutoff=time_cutoff
)

Посмотрим, сколько кандидатов получили из истории прослушиваний для каждого пользователя

In [150]:
hist_sizes = pd.Series({
    uid: len(items)
    for uid, items in hist_recommendations.items()
})

print("Пользователей с историческими кандидатами:", len(hist_recommendations))
print("Среднее количество кандидатов:", hist_sizes.mean())

Пользователей с историческими кандидатами: 8711
Среднее количество кандидатов: 329.3158075995867


Оценка:

In [152]:
# исторические рекомендации в множества item_id
hist_candidates_dict = {
    uid: {item_id for item_id, score in items}
    for uid, items in hist_recommendations.items()
}

user_recalls = []

for uid, relevant_items in valid_relevant.items():
    relevant_items = set(relevant_items)
    if not relevant_items:
        continue

    candidates = hist_candidates_dict.get(uid, set())
    hits = len(candidates & relevant_items)
    user_recalls.append(hits / len(relevant_items))
    
print(f"Recall: {np.mean(user_recalls):.6f}")

Recall: 0.306345


## iALS + BPR + история

Объединим рекомендации iALS, BPR и исторические рекомендации, чтобы увеличить полноту кандидатогенерации

Для объединения результатов нужно привести рекомендации всех методов к единому формату. Исторические рекомендации и результаты iALS и BPR представлены списками, содержащими идентификаторы треков и соответствующие оценки, для объединения нам нужны только идентификаторы треков

In [108]:
def get_item_set(recommendations):
    return {
        item[0] if isinstance(item, (tuple, list)) else item
        for item in recommendations
    }

In [110]:
# Объединяем кандидатов из трёх источников
all_candidates = {}

all_uids = (
    set(hist_candidates_dict)
    | set(ials_recommendations_best)
    | set(bpr_recommendations)
)

for uid in all_uids:
    candidates = set()
    candidates.update(get_item_set(hist_candidates_dict.get(uid, set())))
    candidates.update(get_item_set(ials_recommendations_best.get(uid, [])))
    candidates.update(get_item_set(bpr_recommendations.get(uid, [])))
    all_candidates[uid] = candidates


user_recalls = []

for uid, relevant_items in valid_relevant.items():
    relevant_items = set(relevant_items)
    if relevant_items:
        hits = len(all_candidates.get(uid, set()) & relevant_items)
        user_recalls.append(hits / len(relevant_items))

print(f"Recall: {np.mean(user_recalls):.6f}")

Recall: 0.408957


# 5. Ранжирование

Для ранжирования используем CatBoostRanker

Он будет получать список кандидатов для каждого пользователя и определять, какие треки стоит поставить выше

Сначала обучим модель на кандидатах, полученных с помощью iALS + BPR, потом + история. Для каждой пары пользователь — трек подготовим признаки: оценки iALS, BPR и истории прослушиваний, популярность трека и информацию о предыдущих прослушиваниях этого трека пользователем

## iALS + BPR

In [96]:
from catboost import CatBoostRanker, Pool

In [154]:
# Популярность треков по количеству прослушиваний
item_popularity = (
    train_split[train_split["event_type"] == "listen"]
    .groupby("item_id")
    .size()
)

# Размер истории пользователя
user_history_size = pd.Series(
    np.diff(R.indptr),
    index=user_ids
).to_dict()

In [155]:
def build_ranker_dataset(
    ials_recs,
    bpr_recs,
    valid_relevant,
    top_n=100
):
    rows = []

    for uid, relevant_items in valid_relevant.items():
        relevant_items = set(relevant_items)

        ials = {
            item: (score, rank)
            for rank, (item, score)
            in enumerate(ials_recs.get(uid, [])[:top_n], start=1)
        }

        bpr = {
            item: (score, rank)
            for rank, (item, score)
            in enumerate(bpr_recs.get(uid, [])[:top_n], start=1)
        }

        candidate_items = set(ials) | set(bpr)

        history_size = user_history_size.get(uid, 0)

        for item in candidate_items:
            ials_score, ials_rank = ials.get(item, (0.0, top_n + 1))
            bpr_score, bpr_rank = bpr.get(item, (0.0, top_n + 1))

            rows.append({
                "uid": uid,
                "item_id": item,
                "ials_score": ials_score,
                "bpr_score": bpr_score,
                "ials_rank": ials_rank,
                "bpr_rank": bpr_rank,
                "in_ials": int(item in ials),
                "in_bpr": int(item in bpr),
                "item_popularity": np.log1p(
                    item_popularity.get(item, 0)
                ),
                "user_history_size": np.log1p(history_size),
                "target": int(item in relevant_items)
            })

    return pd.DataFrame(rows)

In [156]:
ranker_data = build_ranker_dataset(
    ials_recommendations_best,
    bpr_recommendations,
    valid_relevant,
    top_n=100
)

Разделим пользователей на обучающую и проверочную выборки в соотношении 80/20. Для этого используем только пользователей, у которых есть хотя бы один релевантный трек

In [99]:
from sklearn.model_selection import train_test_split

user_targets = (
    ranker_data.groupby("uid")["target"]
    .sum()
)

users_with_positive = user_targets[user_targets > 0].index
users_without_positive = user_targets[user_targets == 0].index

train_users, test_users_ranker = train_test_split(
    users_with_positive,
    test_size=0.2,
    random_state=42
)

print("Пользователей для обучения:", len(train_users))
print("Пользователей для проверки:", len(test_users_ranker))
print("Пользователей без позитивов:", len(users_without_positive))

Пользователей для обучения: 5901
Пользователей для проверки: 1476
Пользователей без позитивов: 447


Подготовим данные для CatBoostRanker

Выделим признаки, таргет и объединим треки каждого пользователя в отдельную группу

In [100]:
from catboost import CatBoostRanker, Pool

FEATURES = [
    "ials_score",
    "bpr_score",
    "ials_rank",
    "bpr_rank",
    "in_ials",
    "in_bpr",
    "item_popularity",
    "user_history_size"
]

train_data = ranker_data[
    ranker_data["uid"].isin(train_users)
].copy()

test_data = ranker_data[
    ranker_data["uid"].isin(test_users_ranker)
].copy()

# Группы одного пользователя должны идти подряд
train_data = train_data.sort_values("uid").reset_index(drop=True)
test_data = test_data.sort_values("uid").reset_index(drop=True)

train_pool = Pool(
    data=train_data[FEATURES],
    label=train_data["target"],
    group_id=train_data["uid"]
)

test_pool = Pool(
    data=test_data[FEATURES],
    label=test_data["target"],
    group_id=test_data["uid"]
)

print("Train:", train_data.shape)
print("Test:", test_data.shape)

Train: (1019647, 11)
Test: (255459, 11)


Обучим CatBoostRanker на подготовленных данных

Лосс YetiRank, метрика NDCG@10

In [101]:
ranker = CatBoostRanker(
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    iterations=600,
    learning_rate=0.08,
    depth=8,
    random_seed=42,
    thread_count=4,
    verbose=100,
    early_stopping_rounds=50
)

ranker.fit(
    train_pool,
    eval_set=test_pool,
    use_best_model=True
)

0:	test: 0.1509433	best: 0.1509433 (0)	total: 832ms	remaining: 8m 18s
100:	test: 0.3285933	best: 0.3294937 (95)	total: 1m 8s	remaining: 5m 38s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.3294936775
bestIteration = 95

Shrink model to first 96 iterations.


CatBoostRanker(depth=8, early_stopping_rounds=50, eval_metric='NDCG:top=10', iterations=600, learning_rate=0.08, loss_function='YetiRank', random_seed=42, thread_count=4, verbose=100)

Получилили улучшение NDSG до 0.3294936775

Теперь добавим в список кандидатов исторические рекомендации и проверим, удастся ли еще улучшить результат

## iALS + BPR + история

Рассчитаем признаки истории прослушиваний для каждой пары пользователь-трек: частоту, давность прослушиваний и соответствующие ранги

In [113]:
# Исторические кандидаты
hist_df = pd.DataFrame(
    [
        (uid, item_id)
        for uid, items in hist_candidates_dict.items()
        for item_id in items
    ],
    columns=["uid", "item_id"]
)

# Прослушивания
listens = train_split.loc[
    train_split["event_type"] == "listen",
    ["uid", "item_id", "timestamp"]
].copy()

listens["age"] = (time_cutoff - listens["timestamp"]) / 86400

# Веса свежести
listens["w7"] = np.exp(-listens["age"] / 7)
listens["w30"] = np.exp(-listens["age"] / 30)

# Агрегация признаков
history_features = (
    listens.groupby(["uid", "item_id"], sort=False)
    .agg(
        ui_cnt=("timestamp", "size"),
        ui_last_age=("age", "min"),
        ui_w7=("w7", "sum"),
        ui_w30=("w30", "sum")
    )
    .reset_index()
)

# Ранги
for col in ["ui_w7", "ui_w30", "ui_cnt"]:
    history_features[f"hist_rank_{col}"] = (
        history_features.groupby("uid")[col]
        .rank(method="first", ascending=False)
    )

hist_features = hist_df.merge(
    history_features,
    on=["uid", "item_id"],
    how="left"
)

Объединим кандидатов из iALS, BPR и истории прослушиваний

Для каждой пары пользователь-трек соберем признаки всех источников, добавим популярность трека и размер истории пользователя

Сформируем целевую переменную: 1 - если пользователь прослушал трек на валидации, 0 - нет

In [117]:
# Все кандидаты из трёх источников
candidates_df = pd.DataFrame(
    [
        (uid, item_id)
        for uid, items in all_candidates.items()
        for item_id in items
    ],
    columns=["uid", "item_id"]
)

# Признаки iALS и BPR
model_features = ranker_data[
    [
        "uid", "item_id",
        "ials_score", "bpr_score",
        "ials_rank", "bpr_rank",
        "in_ials", "in_bpr"
    ]
].drop_duplicates(["uid", "item_id"])

# Признаки популярности трека
item_features = ranker_data[
    ["item_id", "item_popularity"]
].drop_duplicates("item_id")

# Признаки истории пользователя
user_features = ranker_data[
    ["uid", "user_history_size"]
].drop_duplicates("uid")

# Объединяем признаки
ranker_data_all = (
    candidates_df
    .merge(model_features, on=["uid", "item_id"], how="left")
    .merge(hist_features, on=["uid", "item_id"], how="left")
    .merge(item_features, on="item_id", how="left")
    .merge(user_features, on="uid", how="left")
)

# Признаки отсутствующих источников заполняем нулями
feature_columns = [
    "ials_score", "bpr_score",
    "ials_rank", "bpr_rank",
    "in_ials", "in_bpr",
    "ui_cnt", "ui_last_age", "ui_w7", "ui_w30",
    "hist_rank_ui_w7", "hist_rank_ui_w30", "hist_rank_ui_cnt",
    "item_popularity", "user_history_size"
]

ranker_data_all[feature_columns] = (
    ranker_data_all[feature_columns].fillna(0)
)

# Целевая переменная
positive_pairs = val_split[["uid", "item_id"]].drop_duplicates()
positive_pairs["target"] = 1

ranker_data_all = ranker_data_all.merge(
    positive_pairs,
    on=["uid", "item_id"],
    how="left"
)

ranker_data_all["target"] = (
    ranker_data_all["target"].fillna(0).astype("int8")
)

print(ranker_data_all.shape)
print(ranker_data_all["target"].sum())

(5993188, 18)
861978


Для обучения CatBoost уменьшим число отрицательных примеров до 100, оставив все положительные, и обучим ранжировщик на полученной выборке

In [119]:
# Сохраняем все положительные примеры
positive_train = train_data_all[
    train_data_all["target"] == 1
]

# Оставляем до 100 отрицательных примеров на пользователя
negative_train = train_data_all[
    train_data_all["target"] == 0
].copy()

rng = np.random.default_rng(42)
negative_train["_random"] = rng.random(len(negative_train))

negative_train = (
    negative_train
    .sort_values(["uid", "_random"])
    .groupby("uid", sort=False)
    .head(100)
    .drop(columns="_random")
)

# Формируем обучающую выборку
train_data_sampled = (
    pd.concat([positive_train, negative_train])
    .sort_values("uid")
)

# Создаём Pool
train_pool_sampled = Pool(
    train_data_sampled[feature_columns],
    label=train_data_sampled["target"],
    group_id=train_data_sampled["uid"]
)

# Валидацию оставляем без изменений
test_pool_all = Pool(
    test_data_all[feature_columns],
    label=test_data_all["target"],
    group_id=test_data_all["uid"]
)

# Обучаем CatBoost
ranker_all = CatBoostRanker(
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    iterations=300,
    learning_rate=0.08,
    depth=6,
    random_seed=42,
    thread_count=4,
    verbose=50,
    early_stopping_rounds=30
)

ranker_all.fit(
    train_pool_sampled,
    eval_set=test_pool_all,
    use_best_model=True
)

0:	test: 0.2836492	best: 0.2836492 (0)	total: 1.15s	remaining: 5m 43s
50:	test: 0.6117059	best: 0.6119476 (47)	total: 58s	remaining: 4m 43s
100:	test: 0.6204845	best: 0.6206501 (91)	total: 1m 54s	remaining: 3m 45s
150:	test: 0.6237343	best: 0.6243577 (148)	total: 2m 49s	remaining: 2m 47s
200:	test: 0.6267075	best: 0.6267075 (200)	total: 3m 44s	remaining: 1m 50s
250:	test: 0.6283496	best: 0.6283496 (250)	total: 4m 40s	remaining: 54.7s
Stopped by overfitting detector  (30 iterations wait)

bestTest = 0.628595527
bestIteration = 264

Shrink model to first 265 iterations.


CatBoostRanker(depth=6, early_stopping_rounds=30, eval_metric='NDCG:top=10', iterations=300, learning_rate=0.08, loss_function='YetiRank', random_seed=42, thread_count=4, verbose=50)

Добавление истории улучшило NDSG до 0.628595527

# 6. Обучение модели на полной выборке

Построим матрицы (обычная и бинарная для bpr) взаимодействий по всей обучающей выборке train:

In [127]:
R_full, user_ids_full, item_ids_full = build_interaction_matrix(train)

R_binary_full = R_full.copy()
R_binary_full.data[:] = 1

user_to_idx_full = {
    uid: idx for idx, uid in enumerate(user_ids_full)
}

print("Матрица:", R_full.shape)

Матрица: (9086, 827802)


Обучим ials и bpr на полной обучающей выборке:

In [158]:
ials_model_full = fit_ials(
    R_full,
    factors=128,
    regularization=0.01,
    iterations=20,
    alpha=10
)

bpr_model_full = BayesianPersonalizedRanking(
    factors=128,
    learning_rate=0.05,
    regularization=0.01,
    iterations=30,
    num_threads=4,
    random_state=42
)

bpr_model_full.fit(R_binary_full)

100%|█████████████████████████████████████████████████| 30/30 [07:14<00:00, 14.50s/it, train_auc=95.11%, skipped=7.79%]


Для каждого тестового пользователя получим по 300 рекомендаций от моделей iALS и BPR. Пользователей, отсутствующих в обучающей выборке, на этом этапе пропускаем, для них позже предусмотрим рекомендации по популярности

In [160]:
known_test_users = [
    uid for uid in test_users["user_id"]
    if uid in user_to_idx_full
]

test_indices = np.array([
    user_to_idx_full[uid] for uid in known_test_users
])

ials_items, ials_scores = ials_model_full.recommend(
    test_indices, R_full[test_indices],
    N=300, filter_already_liked_items=False
)

bpr_items, bpr_scores = bpr_model_full.recommend(
    test_indices, R_binary_full[test_indices],
    N=300, filter_already_liked_items=False
)

Преобразуем результаты обеих моделей в df:

In [162]:
def make_recs_df(users, items, scores, prefix):
    rows = []
    for i, uid in enumerate(users):
        for rank, (idx, score) in enumerate(zip(items[i], scores[i]), 1):
            rows.append((
                uid, int(item_ids_full[idx]),
                float(score), rank, 1
            ))

    return pd.DataFrame(
        rows,
        columns=[
            "uid", "item_id",
            f"{prefix}_score", f"{prefix}_rank", f"in_{prefix}"
        ]
    )

In [163]:
ials_df = make_recs_df(
    known_test_users, ials_items, ials_scores, "ials"
)
bpr_df = make_recs_df(
    known_test_users, bpr_items, bpr_scores, "bpr"
)

In [174]:
ials_df.head()

,uid,item_id,ials_score,ials_rank,in_ials
0,1,858081,1.062065,1,1
1,1,725488,1.055672,2,1
2,1,117033,1.031291,3,1
3,1,380729,1.021277,4,1
4,1,823148,1.019692,5,1


Для тестовых пользователей получим историю прослушиваний:

In [164]:
test_uids = set(test_users["user_id"])

listen_hist = train.loc[
    (train["event_type"] == "listen") & (train["uid"].isin(test_uids)),
    ["uid", "item_id", "timestamp"]
].copy()

In [165]:
listen_hist

,uid,item_id,timestamp
0,1,358453,23850979
1,1,688547,23851159
2,1,19131,23851564
3,1,798591,23851719
4,1,557996,23851889
...,...,...,...
43000342,10000,908685,47562079
43000343,10000,466499,47638994
43000344,10000,582137,47639359
43000345,10000,582137,47639674


Рассчитаем, как давно был прослушан трек относительно самого позднего времени в в train:

In [166]:
cutoff = train["timestamp"].max()
listen_hist["age"] = (cutoff - listen_hist["timestamp"]) / 86400

Добавим веса, которые будут уменьшать значимость прослушивания пропорц-но давности прослушивания

Используем экспонен. затухание

In [168]:
listen_hist["w7"] = np.exp(-listen_hist["age"] / 7)
listen_hist["w30"] = np.exp(-listen_hist["age"] / 30)

Группируем события по (uid, item_id) и считаем для каждой пары 4 хар-ки: кол-во прослушиваний, давность последнего прослушивания, суммарный вес за 7 и 30 дней

In [169]:
hist_full = (
    listen_hist.groupby(["uid", "item_id"], sort=False)
    .agg(
        ui_cnt=("timestamp", "size"),
        ui_last_age=("age", "min"),
        ui_w7=("w7", "sum"),
        ui_w30=("w30", "sum")
    )
    .reset_index()
)

In [172]:
hist_full.head()

,uid,item_id,ui_cnt,ui_last_age,ui_w7,ui_w30,hist_rank_ui_w7,hist_rank_ui_w30,hist_rank_ui_cnt
0,1,358453,1,278.863194,4.997473e-18,0.000092,269.0,269.0,86.0
1,1,688547,1,278.861111,4.998961e-18,0.000092,268.0,268.0,87.0
2,1,19131,1,278.856424,5.002310e-18,0.000092,267.0,267.0,88.0
3,1,798591,1,278.854630,5.003592e-18,0.000092,266.0,266.0,89.0
4,1,557996,1,278.852662,5.004998e-18,0.000092,265.0,265.0,90.0


Ранждируем треки внутри истории каждого юзера:

In [170]:
for col in ["ui_w7", "ui_w30", "ui_cnt"]:
    hist_full[f"hist_rank_{col}"] = (
        hist_full.groupby("uid")[col]
        .rank(method="first", ascending=False)
    )

Отбираем топ 300 треков из 7дю истории, 150 из 30д истории и 100 по числу прослушиваний:

In [171]:
hist_full = hist_full[
    (hist_full["hist_rank_ui_w7"] <= 300) |
    (hist_full["hist_rank_ui_w30"] <= 150) |
    (hist_full["hist_rank_ui_cnt"] <= 100)
]

Объединяем кандидатов из ials, bpr и истории прослушиваний (оставляем только uid и item_id), удалим дубликаты:

In [173]:
candidates = pd.concat([
    hist_full[["uid", "item_id"]],
    ials_df[["uid", "item_id"]],
    bpr_df[["uid", "item_id"]]
]).drop_duplicates()

Добавим к кандидатам оценки ials, bpr и признаки из истории:

In [176]:
test_features = (
    candidates
    .merge(ials_df, on=["uid", "item_id"], how="left")
    .merge(bpr_df, on=["uid", "item_id"], how="left")
    .merge(hist_full, on=["uid", "item_id"], how="left")
)

Рассчитаем популярность каждого трека по всей обучающей выборке

В качестве показателя используем логарифм количества событий прослушивания данного трека (чтобы уменьшить различие между очень популярными и не популярными треками)

In [177]:
pop = (
    train[train["event_type"] == "listen"]
    .groupby("item_id").size().map(np.log1p)
    .rename("item_popularity")
)


Рассчитаем размер истории каждого пользователя как количество уникальных треков, которые он прослушал в обучающей выборке. К полученному значению также применим логариф

Этот признак характеризует активность пользователя и позволяет ранжировщику учитывать различия между пользователями с короткой и длинной историей

In [178]:
uh = (
    train[train["event_type"] == "listen"]
    .groupby("uid")["item_id"].nunique().map(np.log1p)
    .rename("user_history_size")
)

Добавим новые признаки к test_features:

In [179]:
test_features = test_features.merge(pop, on="item_id", how="left")
test_features = test_features.merge(uh, on="uid", how="left")

## Ранжирование

Перед применением ранжировщика заменим пропущенные значения признаков нулями

In [180]:
test_features[feature_columns] = test_features[feature_columns].fillna(0)

Применяем обученный ранжировщик CatBoostRanker:

In [181]:
test_features["score"] = ranker_all.predict(test_features[feature_columns])

Выбор 10 лучших рекомендаций:

In [182]:
top10 = (
    test_features
    .sort_values(["uid", "score"], ascending=[True, False])
    .groupby("uid")
    .head(10)
)

Для пользователей без истории предсказываем поплуярные треки:

In [183]:
popular = (
    train[train["event_type"] == "listen"]
    .groupby("item_id").size().nlargest(10).index.tolist()
)

## Сабмит

In [184]:
pred = (
    top10.groupby("uid")["item_id"]
    .apply(lambda x: " ".join(map(str, x.tolist())))
    .to_dict()
)

fallback = " ".join(map(str, popular))

In [185]:
submission = pd.DataFrame({
    "user_id": test_users["user_id"],
    "item_ids": [pred.get(uid, fallback) for uid in test_users["user_id"]]
})

submission_final.to_csv("submission.csv", index=False)